# ColabProSST automated release acceptance

This notebook runs ColabProSST through its real production workflow APIs and creates a downloadable CSV, JSON, Markdown, and ZIP acceptance report. It does not replace the final browser checks listed at the bottom.

## Before running

1. Select **Runtime > Change runtime type > GPU**. An A100 is recommended for the full profile; L4 or T4 can take substantially longer.
2. Run the single code cell below. On a new runtime, installation restarts the Python kernel once. After Colab reconnects, run the same cell one more time.
3. Keep this tab connected until the final report is produced. The full profile downloads and sequentially tests all six official ProSST models, so it can take a long time.
4. To test Hugging Face upload, create a new target repository name, add a write token as the Colab Secret `HF_TOKEN`, enable notebook access for that secret, and set `RUN_HF_UPLOAD` below. The upload check is optional and disabled by default.

## Profiles

- **full:** all workflows plus real weight loading and one `input_ids + ss_input_ids` forward pass for ProSST-20, 128, 512, 1024, 2048, and 4096.
- **core:** all workflows, training modes, and prediction tasks, but only metadata/tokenizer checks for the six-model family. Use this first on a free T4 runtime.


In [ ]:
#@title Run ColabProSST acceptance
PROFILE = 'full' #@param ['core', 'full']
REQUIRE_GPU = True #@param {type:'boolean'}
RUN_HF_UPLOAD = False #@param {type:'boolean'}
HF_REPO_ID = '' #@param {type:'string'}
HF_PRIVATE = True #@param {type:'boolean'}
DOWNLOAD_REPORT = True #@param {type:'boolean'}

import getpass
import json
import os
import sys
import urllib.request
from pathlib import Path

PRODUCT_NOTEBOOK_URL = (
    'https://raw.githubusercontent.com/westlake-repl/SaprotHub/'
    'main/colab/ColabProSST.ipynb'
)
product_notebook_path = Path('/content/SaprotHub/colab/ColabProSST.ipynb')
if product_notebook_path.is_file():
    product_notebook = json.loads(
        product_notebook_path.read_text(encoding='utf-8')
    )
else:
    with urllib.request.urlopen(PRODUCT_NOTEBOOK_URL, timeout=60) as response:
        product_notebook = json.loads(response.read().decode('utf-8'))

bootstrap_source = next(
    ''.join(cell['source'])
    for cell in product_notebook['cells']
    if cell.get('cell_type') == 'code'
    and 'ENVIRONMENT_GENERATION' in ''.join(cell.get('source', []))
    and 'COLABPROSST_UI.launch()' in ''.join(cell.get('source', []))
)
os.environ['COLABPROSST_SKIP_UI'] = '1'
exec(compile(bootstrap_source, 'ColabProSST.ipynb', 'exec'), globals())

if RUN_HF_UPLOAD:
    from huggingface_hub import get_token, login

    if get_token() is None:
        token = ''
        try:
            from google.colab import userdata

            token = userdata.get('HF_TOKEN') or ''
        except Exception:
            pass
        if not token:
            token = getpass.getpass('Hugging Face write token: ')
        if not token:
            raise RuntimeError('Hugging Face upload requires a write token.')
        login(token=token, add_to_git_credential=False)

from saprot.utils.colab_prosst_acceptance import (
    ColabProSSTAcceptanceRunner,
)

runner = ColabProSSTAcceptanceRunner(
    profile=PROFILE,
    require_gpu=REQUIRE_GPU,
    run_hf_upload=RUN_HF_UPLOAD,
    hf_repo_id=HF_REPO_ID,
    hf_private=HF_PRIVATE,
)
acceptance = runner.run()

import pandas as pd
from IPython.display import Markdown, display

display(Markdown(Path(acceptance['report_markdown']).read_text(encoding='utf-8')))
display(pd.read_csv(acceptance['report_csv']))
print('Acceptance report ZIP:', acceptance['report_zip'])
if DOWNLOAD_REPORT:
    from google.colab import files

    files.download(acceptance['report_zip'])
if not acceptance['success']:
    raise RuntimeError(
        'Required acceptance checks failed: '
        + ', '.join(acceptance['required_failures'])
    )
print('All required ColabProSST acceptance checks passed.')


## Final manual browser checks

After the automated report passes, open the normal [ColabProSST notebook](https://colab.research.google.com/github/westlake-repl/SaprotHub/blob/main/colab/ColabProSST.ipynb) and confirm these browser-only behaviors:

- Each result button starts exactly one local download.
- The interface remains visible after a long task and its result buttons remain clickable.
- Go back returns to the previous menu and Refresh resets only the current page.
- A real upload appears at the expected Hugging Face URL when the optional upload check is enabled.
